# ML Training Workflow

End-to-end LightGBM training using pipeline-generated features and triple-barrier labels.

1. **Setup** - Configure markets, date range, and label parameters
2. **Load Data** - Read features and labels from parquet store
3. **Explore Data** - Check distributions, missing values, correlations
4. **Preprocess** - Handle NaN, split train/test
5. **Train** - LightGBM with early stopping
6. **Evaluate** - Metrics, ROC, threshold analysis
7. **Feature Importance** - Identify top predictors

## Setup

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from training import Dataset, LGBMTrainer, config
from qate.env import sys_env
from qate.env.env import Env
from qate.env.env_name import EnvName
from qate.store.parquet_store import ParquetStore
from qate.util.dt_range import DtRange

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
sns.set_style('whitegrid')
%matplotlib inline

env = Env.load_env(sys_env.ENV_DIR, EnvName.MEGATRON)
env.visit()

EXEC_CROSS_VALIDATION = False

(
    FEATURE_MARKET,
    LABEL_MARKET,
    EXECUTION_SIZE,
    TARGET_COLUMN,
    CONFIG,
) = config.setup()

# DATE RANGE
DATA_START_DATE = "20250811"
DATA_END_DATE = "20250814"
DT_RANGE = DtRange.from_strings(DATA_START_DATE, DATA_END_DATE)


## 1. Load Data

Read features and labels from parquet, normalize timezones.

In [ ]:
parquet_store = ParquetStore(sys_env.get_parquet_root_dir())

# Load features
feature_store = parquet_store.create_feature(FEATURE_MARKET)
features_df = feature_store.read(DT_RANGE)

# Load labels (param_id must match the pipeline run)
label_store = parquet_store.create_label(
      "TripleBarrierLabel_tmp",
      LABEL_MARKET,
      CONFIG.param_id,
)
# label_store = parquet_store.create_label('TripleBarrierLabel', LABEL_MARKET, CONFIG.param_id)
labels_df = label_store.read(DT_RANGE)

print(f'Features: {features_df.shape[0]:,} rows x {features_df.shape[1]} columns')
print(f'Labels:   {labels_df.shape[0]:,} rows x {labels_df.shape[1]} columns')

def prepare_df(df):
    """Ensure DatetimeIndex in UTC, handle ts column if present."""
    if 'ts' in df.columns:
        df.index = pd.to_datetime(df['ts'], unit='s')
        df = df.drop(columns=['ts'])
    return df.sort_index()

features_df = prepare_df(features_df)
labels_df = prepare_df(labels_df)

print(f'\nFeatures range: {features_df.index.min()} -> {features_df.index.max()}')
print(f'Labels range:   {labels_df.index.min()} -> {labels_df.index.max()}')

# Original label distribution
print('\nTriple-barrier label distribution:')
print(labels_df[TARGET_COLUMN].value_counts().sort_index().to_string())

display(features_df.head())
display(labels_df.head())


### 1.1 Align Features and Labels, Filter by Vol/Spread

1. Align features with labels using `merge_asof` (nearest timestamp matching)
2. Include `volatility_ema` and `spread_pct_ema` in the merged result
3. Filter to keep only rows where `volatility_ema > spread_pct_ema * min_ratio`

This ensures each feature is matched to its true nearest label before filtering.

In [ ]:
# Create Dataset and align features with labels
# Include volatility_ema and spread_pct_ema for post-alignment filtering
dataset = Dataset(
    features_df=features_df,
    labels_df=labels_df,
    target_column=TARGET_COLUMN,
)

# Align and include filter columns
dataset.align(include_label_columns=['volatility_ema', 'spread_pct_ema'])

X, y = dataset.get_aligned_data()
print(f'After alignment: {X.shape[0]:,} rows x {X.shape[1]} columns')
print(f'  Label distribution (0=Short, 1=Long, 2=Timeout): {y.value_counts().sort_index().to_dict()}')

# Preserve original triple-barrier labels for later plotting
y_tb = y.copy()

# Filter by vol/spread ratio
dataset.filter_by_vol_spread(min_ratio=CONFIG.min_vol_spread_ratio)

X, y = dataset.get_aligned_data()
print(f'
After vol/spread filtering (ratio={CONFIG.min_vol_spread_ratio}):')
print(f'  {X.shape[0]:,} rows x {X.shape[1]} features')
print(f'  Label distribution (0=Short, 1=Long, 2=Timeout): {y.value_counts().sort_index().to_dict()}')

# Preserve filtered triple-barrier labels for plotting
y_tb_filtered = y.copy()


## 2. Explore Data

In [ ]:
# Missing values
na_pct = (X.isna().sum() / len(X) * 100).sort_values(ascending=False)
na_nonzero = na_pct[na_pct > 0]

if len(na_nonzero) > 0:
    print(f'{len(na_nonzero)} columns with missing values (top 20):')
    print(na_nonzero.head(20).to_string())
else:
    print('No missing values.')


In [ ]:
# Feature statistics
X.describe().T


In [ ]:
# Target distribution

from training.plot_labels import plot_labels
plot_labels(y_tb, CONFIG)

plot_labels(y_tb_filtered, CONFIG)


In [ ]:
# Feature correlation with target (top 20)
correlations = X.corrwith(y).abs().sort_values(ascending=False)
print('Top 20 features correlated with target:')
print(correlations.head(20).to_string())


In [ ]:
# Correlation heatmap of top features
top_features = correlations.head(15).index.tolist()

plt.figure(figsize=(12, 10))
corr_matrix = X[top_features].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Feature Correlation Matrix (Top 15 by Target Correlation)')
plt.tight_layout()
plt.show()


## 3. Preprocess Data

In [ ]:
# Leak-safe preprocessing: split first, then clean based on Train statistics
# This replaces drop_na_features and fill_na calls
dataset.leak_safe_preprocess(test_size=0.2, na_threshold=0.5, fill_method='ffill')

X_train, y_train = dataset.get_train_data()
X_test, y_test = dataset.get_test_data()

print(f'After leak-safe preprocessing:')
print(f'Train: {len(X_train):,} samples, {len(X_train.columns)} features')
print(f'Test:  {len(X_test):,} samples, {len(X_test.columns)} features')

# Align original triple-barrier labels to the post-preprocessing feature index
# We use union because the data is now split
y_tb = y_tb.loc[X_train.index.union(X_test.index)]


## 4. Train Model

In [ ]:
print(f'Train: {len(X_train):,} samples  ({X_train.index.min()} -> {X_train.index.max()})')
print(f'Test:  {len(X_test):,} samples  ({X_test.index.min()} -> {X_test.index.max()})')

# Multiclass classification: 0=Short, 1=Long, 2=Timeout
trainer = LGBMTrainer(
    task='classification',
    params={
        'objective': 'multiclass',
        'num_class': 3,
        'metric': 'multi_logloss',
        'boosting_type': 'gbdt',
        'num_leaves': 63,
        'max_depth': 8,
        # 'min_data_in_leaf': 10,
        'learning_rate': 0.05,
        'feature_fraction': 0.9,
        'bagging_fraction': 0.8,
        'bagging_freq': 5,
        'verbose': 1,
        'n_jobs': -1,
        'random_state': 42,
    },
    num_boost_round=1000,
    early_stopping_rounds=50,
    verbose_eval=100,
)

trainer.fit(
    X_train, y_train,
    X_val=X_test, y_val=y_test,
)

print(f'\nBest iteration: {trainer.best_iteration}')


In [ ]:
# Training history
if trainer.training_history:
    fig, ax = plt.subplots(figsize=(10, 5))
    for name, metrics in trainer.training_history.items():
        for metric, values in metrics.items():
            ax.plot(values, label=f'{name}_{metric}')
    if trainer.best_iteration:
        ax.axvline(x=trainer.best_iteration, color='r', linestyle='--',
                   label=f'Best iter ({trainer.best_iteration})')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Loss')
    ax.set_title('Training History')
    ax.legend()
    plt.tight_layout()
    plt.show()


## 5. Evaluate Model

In [ ]:
# 5.1 Multiclass evaluation
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Get predictions (multiclass returns probabilities for each class)
y_proba = trainer.predict(X_test)  # Shape: (n_samples, 3)
y_pred = np.argmax(y_proba, axis=1)

# Class names for display
class_names = ['Short (0)', 'Long (1)', 'Timeout (2)']

print('='*60)
print('Classification Report')
print('='*60)
print(classification_report(y_test, y_pred, target_names=class_names))

print(f'\nOverall Accuracy: {accuracy_score(y_test, y_pred):.4f}')


In [ ]:
# 5.2 Confusion matrix (3x3 for multiclass)
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Pred Short', 'Pred Long', 'Pred Timeout'],
    yticklabels=['Actual Short', 'Actual Long', 'Actual Timeout'],
)
plt.title('Confusion Matrix (3-class)')
plt.tight_layout()
plt.show()

# Show class-wise accuracy
print('\nClass-wise accuracy:')
for i, name in enumerate(class_names):
    class_mask = y_test == i
    if class_mask.sum() > 0:
        class_acc = (y_pred[class_mask] == i).mean()
        print(f'  {name}: {class_acc:.2%} ({class_mask.sum():,} samples)')


In [ ]:
# 5.3 Trading simulation
from simulation.v0 import simulate_trading, plot_simulation_results

sim_result = simulate_trading(
    predictions=y_pred,
    labels_df=labels_df,
    test_indices=X_test.index,
    
)

sim_result.print_summary()

plot_simulation_results(sim_result, window=500)


In [ ]:
# 5.4 Simulation v2: leak-safe alignment + non-overlapping trades
from simulation.v1 import simulate_trading_strict, plot_simulation_results

sim_result = simulate_trading_strict(
    predictions=y_pred,
    labels_df=labels_df,
    test_indices=X_test.index,
    timeout_sec=CONFIG.timeout_sec,
    allow_overlap=False,
    probabilities=y_proba,
    threshold=0.4,
)
sim_result.print_summary()
plot_simulation_results(sim_result)


In [ ]:
# 5.5 Confidence vs accuracy plot
# Build analysis frame if not already created
confidence = y_proba.max(axis=1)
correct = (y_pred == y_test.values)
analysis_df = pd.DataFrame({
    'y_true': y_test.values,
    'y_pred': y_pred,
    'confidence': confidence,
    'correct': correct,
})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy vs confidence threshold
thresholds = np.arange(0.35, 0.95, 0.05)
accs, coverages = [], []
for thresh in thresholds:
    mask = confidence >= thresh
    if mask.sum() > 0:
        accs.append(correct[mask].mean())
        coverages.append(mask.mean())
    else:
        accs.append(np.nan)
        coverages.append(0)

ax1 = axes[0]
ax2 = ax1.twinx()
ax1.plot(thresholds, accs, 'b-', marker='o', label='Accuracy')
ax2.plot(thresholds, coverages, 'r--', marker='s', label='Coverage')
ax1.set_xlabel('Confidence Threshold')
ax1.set_ylabel('Accuracy', color='b')
ax2.set_ylabel('Coverage (fraction of samples)', color='r')
ax1.set_title('Accuracy vs Coverage Trade-off')
ax1.legend(loc='lower left')
ax2.legend(loc='lower right')
ax1.grid(True, alpha=0.3)


# Confidence bucket summary
bucket_edges = [0.0, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
analysis_df['conf_bucket'] = pd.cut(analysis_df['confidence'], bucket_edges, right=False)

bucket_summary = (
    analysis_df.groupby('conf_bucket')
    .agg(samples=('confidence', 'size'), accuracy=('correct', 'mean'))
    .reset_index()
)

print('Confidence bucket summary:')
print(bucket_summary.to_string(index=False, formatters={'accuracy': '{:.2%}'.format}))

# Prediction distribution by actual class
pred_by_true = pd.crosstab(analysis_df['y_true'], analysis_df['y_pred'], normalize='index')
pred_by_true.plot(kind='bar', ax=axes[1], color=['red', 'orange', 'green'])
axes[1].set_xlabel('Actual Class')
axes[1].set_ylabel('Fraction')
axes[1].set_title('Prediction Distribution by Actual Class')
axes[1].set_xticklabels(['Short', 'Long', 'Timeout'], rotation=0)
axes[1].legend(['Pred Short', 'Pred Long', 'Pred Timeout'])

plt.tight_layout()
plt.show()


## 6. Feature Importance

In [ ]:
importance_df = trainer.feature_importance(importance_type='gain')
print('Top 20 important features:')
importance_df.head(20)


In [ ]:
top_n = min(20, len(importance_df))
top_imp = importance_df.head(top_n)

plt.figure(figsize=(10, 8))
plt.barh(range(top_n), top_imp['importance'].values)
plt.yticks(range(top_n), top_imp['feature'].values)
plt.xlabel('Importance (Gain)')
plt.title(f'Top {top_n} Feature Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


## 7. Prediction Analysis

In [ ]:
# 7.1 Prediction analysis summary

# Prediction counts
print('\nPrediction distribution:')
for i, name in enumerate(class_names):
    count = (y_pred == i).sum()
    pct = count / len(y_pred) * 100
    print(f'  {name}: {count:,} ({pct:.1f}%)')

# When model predicts Long, what's the actual distribution?
print('\nWhen model predicts Long (class 1):')
long_mask = y_pred == 1
if long_mask.sum() > 0:
    actual_dist = y_test[long_mask].value_counts().sort_index()
    for cls, count in actual_dist.items():
        print(f'  Actual {class_names[cls]}: {count:,} ({count/long_mask.sum()*100:.1f}%)')

# When model predicts Short, what's the actual distribution?
print('\nWhen model predicts Short (class 0):')
short_mask = y_pred == 0
if short_mask.sum() > 0:
    actual_dist = y_test[short_mask].value_counts().sort_index()
    for cls, count in actual_dist.items():
        print(f'  Actual {class_names[cls]}: {count:,} ({count/short_mask.sum()*100:.1f}%)')


In [ ]:
# 7.2 Probability distributions by actual class
# Define colors if not already present
colors = ['red', 'orange', 'green']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, (name, color) in enumerate(zip(class_names, colors)):
    ax = axes[i]
    for j, (actual_name, actual_color) in enumerate(zip(class_names, colors)):
        mask = y_test == j
        ax.hist(y_proba[mask, i], bins=30, alpha=0.5, color=actual_color, 
                label=f'Actual {actual_name}', density=True)
    ax.set_xlabel(f'P({name})')
    ax.set_ylabel('Density')
    ax.set_title(f'P({name}) by Actual Class')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
# 7.3 Predictions over time
analysis_ts = analysis_df.copy()
analysis_ts.index = X_test.index

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
window = 500

# Rolling accuracy
rolling_acc = analysis_ts['correct'].rolling(window).mean()
axes[0].plot(analysis_ts.index, rolling_acc, color='blue')
axes[0].axhline(y=1/3, color='r', linestyle='--', alpha=0.5, label='Random (33%)')
axes[0].set_ylabel('Accuracy')
axes[0].set_title(f'Rolling Accuracy (window={window})')
axes[0].legend()
axes[0].set_ylim(0, 0.6)

# Rolling class predictions
for i, (name, color) in enumerate(zip(class_names, colors)):
    pred_rate = (analysis_ts['y_pred'] == i).rolling(window).mean()
    axes[1].plot(analysis_ts.index, pred_rate, color=color, label=name, alpha=0.7)
axes[1].set_ylabel('Prediction Rate')
axes[1].set_title(f'Rolling Prediction Distribution (window={window})')
axes[1].legend()

# Rolling confidence
rolling_conf = analysis_ts['confidence'].rolling(window).mean()
axes[2].plot(analysis_ts.index, rolling_conf, color='purple')
axes[2].axhline(y=1/3, color='r', linestyle='--', alpha=0.5, label='Random baseline')
axes[2].set_ylabel('Confidence')
axes[2].set_title(f'Rolling Mean Confidence (window={window})')
axes[2].set_xlabel('Time')

plt.tight_layout()
plt.show()


## 8. Cross-Validation (Optional)

In [ ]:
# Cross-validation for multiclass
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import accuracy_score, f1_score

def cross_validation():
    X_all, y_all = dataset.get_aligned_data()

    tscv = TimeSeriesSplit(n_splits=5)
    cv_results = {'accuracy': [], 'f1_macro': [], 'f1_weighted': []}

    for fold, (train_idx, val_idx) in enumerate(tscv.split(X_all)):
        X_cv_train, X_cv_val = X_all.iloc[train_idx], X_all.iloc[val_idx]
        y_cv_train, y_cv_val = y_all.iloc[train_idx], y_all.iloc[val_idx]
        
        # Train a fresh model for this fold
        cv_trainer = LGBMTrainer(
            task='classification',
            params={
                'objective': 'multiclass',
                'num_class': 3,
                'metric': 'multi_logloss',
                'boosting_type': 'gbdt',
                'num_leaves': 31,
                'learning_rate': 0.05,
                'verbose': -1,
                'n_jobs': -1,
            },
            num_boost_round=200,
            early_stopping_rounds=30,
            verbose_eval=0,
        )
        cv_trainer.fit(X_cv_train, y_cv_train, X_val=X_cv_val, y_val=y_cv_val)
        
        # Predict
        y_proba_cv = cv_trainer.predict(X_cv_val)
        y_pred_cv = np.argmax(y_proba_cv, axis=1)
        
        # Metrics
        cv_results['accuracy'].append(accuracy_score(y_cv_val, y_pred_cv))
        cv_results['f1_macro'].append(f1_score(y_cv_val, y_pred_cv, average='macro'))
        cv_results['f1_weighted'].append(f1_score(y_cv_val, y_pred_cv, average='weighted'))
        
        print(f'Fold {fold+1}: Acc={cv_results["accuracy"][-1]:.4f}, F1_macro={cv_results["f1_macro"][-1]:.4f}')

    cv_summary = pd.DataFrame(cv_results)
    print('\nCross-Validation Summary:')
    print(cv_summary.describe().T[['mean', 'std']].to_string())

    fig, ax = plt.subplots(figsize=(10, 5))
    cv_summary.plot(kind='box', ax=ax)
    ax.axhline(y=1/3, color='r', linestyle='--', alpha=0.5, label='Random baseline')
    ax.set_ylabel('Score')
    ax.set_title('Cross-Validation Results (Multiclass)')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

if EXEC_CROSS_VALIDATION:
    cross_validation()


## 9. Save Model

In [ ]:
MODEL_PATH = '../models/lgbm_model'
trainer.save(MODEL_PATH)
print(f'Model saved to {MODEL_PATH}')

# Verify load
loaded_trainer = LGBMTrainer.load(MODEL_PATH)
original_preds = trainer.predict(X_test)
loaded_preds = loaded_trainer.predict(X_test)

match = np.allclose(original_preds, loaded_preds)
print(f'Predictions match after reload: {match}')
